# Survey property maps with the LSST Butler

Contact author: Alex Broughton
<br>Date: 


In [ ]:
! eups list -s | grep lsst_distrib


## Introduction

**Consolidated survey property maps** summarize quantities such as exposure time, PSF size, sky level, and magnitude limits onto the sphere. In Rubin pipelines they are stored as **HealSparse** datasets: each HEALPix cell holds one statistic per sky location.

This notebook is a hands-on tutorial. You will learn how to:

- **Discover** which `deepCoadd_*_consolidated_map_*` products exist in a Butler repository.
- **Probe footprint and native resolution** quickly using the `.coverage` dataset.
- **Load a degraded map** so `butler.get` stays practical for interactive work.
- **Map RA and Dec to nested HEALPix pixel indices** that match the map you loaded.
- **Read pixel values** at sky positions with `HealSparseMap.get_values_pos`.

**Prerequisites:** a Rubin LSST Science Pipelines stack with `lsst`, `healsparse`, `healpy`, and (for projected plots) `skyproj`; a Butler repo config and collection that contain consolidated survey property maps. Edit the tutorial knobs in the first code cell (`CONFIG`, `COLLECTIONS`, `SKYMAP`, `BAND`, `DEGRADE_NSIDE`) to match your site and run.

> **Heads-up:** Native maps can use very large $N_\mathrm{side}$ (for example 32768). For exploration, pass `parameters={"degrade_nside": ...}` on every value-map read unless you truly need full resolution.


## 1.0 Set Up


In [ ]:
### Import packages + setup common variables and useful functions
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp

import skyproj

from lsst.daf.butler import Butler
import lsst.geom as geom

from astropy.visualization import ZScaleInterval, LinearStretch, ImageNormalize

# For static HTML or publication exports, keep inline. For interactive pan/zoom in Jupyter, try:
# %matplotlib widget
%matplotlib inline

plt.style.use("tableau-colorblind10")

# --- Tutorial knobs (edit these) ---
CONFIG = "dp2_prep"
COLLECTIONS = "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3"
SKYMAP = "lsst_cells_v2"
BAND = "i"

# Degradation: lower NSIDE => faster read, coarser HEALPix cells.
# Must be a power of 2 and <= native nside_sparse reported by .coverage.
DEGRADE_NSIDE = 1024

# Example sky positions (degrees) used later in the pixel + value walkthrough
TARGET_RA_DEG = 60.0
TARGET_DEC_DEG = -37.0


def ra_dec_to_nested_healpix(ra_deg, dec_deg, nside: int) -> np.ndarray:
    """Nested HEALPix indices for Rubin consolidated maps at the given NSIDE."""
    return hp.ang2pix(nside, ra_deg, dec_deg, lonlat=True, nest=True)


def nested_healpix_center_lonlat(pixel: int, nside: int) -> tuple[float, float]:
    """Lon, lat in degrees at the center of a nested HEALPix pixel."""
    lon, lat = hp.pix2ang(nside, int(pixel), nest=True, lonlat=True)
    return float(lon), float(lat)


## 2.0 Construct the Butler

Point `COLLECTIONS` at the DRP run (or tagged collection) that produced the consolidated maps you care about. If you are unsure what is available, `butler.registry.queryCollections()` is your friend.

In [ ]:
butler = Butler(CONFIG, collections=COLLECTIONS)


## 3.0 Discover consolidated map dataset types

Science maps follow the pattern `deepCoadd_<quantity>_consolidated_map_<statistic>`. Separate `propertyMapSurvey_*_SurveyWidePropertyMapPlot` entries are **QC plot** datasets, not the HealSparse value maps used for analysis.

In [ ]:
for dtype in sorted(butler.registry.queryDatasetTypes(expression="*consolidated_map*")):
    print(dtype.name)


### Reference: common `deepCoadd_*` value maps

| Dataset (names vary slightly by pipeline) | Quantity | Units |
|-------------------------------------------|----------|-------|
| `deepCoadd_exposure_time_consolidated_map_sum` | Total exposure time | s |
| `deepCoadd_epoch_consolidated_map_min` / `max` / `mean` | Epoch statistics | MJD |
| `deepCoadd_psf_size_consolidated_map_weighted_mean` | PSF width (det radius) | pixel |
| `deepCoadd_psf_e1_consolidated_map_weighted_mean`, `..._e2_...` | PSF ellipticity | — |
| `deepCoadd_psf_maglim_consolidated_map_weighted_mean` | 5σ PSF flux mag limit | mag (AB) |
| `deepCoadd_sky_background_consolidated_map_weighted_mean` | Mean sky level | nJy |
| `deepCoadd_sky_noise_consolidated_map_weighted_mean` | Sky noise | nJy |
| `deepCoadd_dcr_*_consolidated_map_weighted_mean` | DCR-related summaries | — |

## 4.0 Coverage first: cheap metadata before the heavy map

The **coverage** dataset uses the suffix `.coverage`. It is small and quick to `get`, and it tells you:

- Where on the sky the sparse map has entries
- The **native** `nside_sparse` used when maps are written at full resolution

Coverage is **not** a science-value map: there is no `get_values_pos` on this object.

In [ ]:
coverage = butler.get(
    "deepCoadd_exposure_time_consolidated_map_sum.coverage",
    band=BAND,
    skymap=SKYMAP,
)
print(f"Native nside_sparse from coverage: {coverage.nside_sparse}")
coverage


> **Tip:** After you load a **degraded** value map, always use **`value_map.nside_sparse`** (not the coverage object's native NSIDE) when converting RA/Dec to pixels for queries against that in-memory map.

## 5.0 Load a degraded value map (`HealSparseMap`)

### Why `degrade_nside`?

A full-resolution consolidated map can require a long `butler.get` and a lot of RAM. Passing:

```python
parameters={"degrade_nside": DEGRADE_NSIDE}
```

asks the Butler to **downsample on read** to a coarser HEALPix grid. That is the usual way to make survey-wide maps pleasant for tutorials and exploratory analysis.

- If reads are still slow, try **512** or **256**.
- If you need finer detail, increase toward the native `nside_sparse` from coverage (still powers of two).

Below we load **i-band total exposure time** (`..._exposure_time_..._sum`) as the running example.

In [ ]:
DEGRADE_NSIDE = 512

In [ ]:
hsp_exptime = butler.get(
    "deepCoadd_exposure_time_consolidated_map_sum",
    band=BAND,
    skymap=SKYMAP,
    parameters={"degrade_nside": DEGRADE_NSIDE},
)
hsp_exptime


## 6.0 Pixels for specific RA and Dec

Rubin consolidated maps use **nested** HEALPix ordering. The rule is simple: **use the same `nside` as `hsp_exptime.nside_sparse`**, which equals `DEGRADE_NSIDE` immediately after a degraded read.

1. `hp.ang2pix(nside, ra, dec, lonlat=True, nest=True)` gives the **nested** pixel index (or indices) for your coordinate(s).
2. `hp.pix2ang(nside, pixel, nest=True, lonlat=True)` returns the **pixel center** on the sphere.

The next cell ties a **single** `(RA, Dec)` example to its pixel and exposure time.

In [ ]:
nside = int(hsp_exptime.nside_sparse)
pix_target = int(ra_dec_to_nested_healpix(TARGET_RA_DEG, TARGET_DEC_DEG, nside))
lon_c, lat_c = nested_healpix_center_lonlat(pix_target, nside)
exptime_target = float(hsp_exptime.get_values_pos(TARGET_RA_DEG, TARGET_DEC_DEG))

print(f"NSIDE used for this map: {nside}")
print(f"Target RA, Dec (deg): {TARGET_RA_DEG}, {TARGET_DEC_DEG}")
print(f"Nested HEALPix pixel: {pix_target}")
print(f"Pixel center lon, lat (deg): {lon_c:.6f}, {lat_c:.6f}")
print(f"Exposure time at target (s): {exptime_target}")


### Vectorized RA/Dec arrays

NumPy broadcasting rules apply: you can pass arrays of RA and Dec with the same shape, or combinations that broadcast together.

In [ ]:
ra_deg = np.linspace(150.0, 300.0, 5)
dec_deg = np.linspace(-30.0, -10.0, 5)
pixels = ra_dec_to_nested_healpix(ra_deg, dec_deg, nside)
print("RA (deg):", ra_deg)
print("Dec (deg):", dec_deg)
print("Nested HEALPix indices:", pixels)


## 7.0 Survey property values at sky positions

On a `HealSparseMap`, call **`get_values_pos(ra, dec)`** with angles in **degrees**. Results follow NumPy broadcasting.

Off the survey footprint, HealSparse returns a **sentinel** fill value. Mask with `np.isfinite` before taking means or percentiles.

In [ ]:
val = hsp_exptime.get_values_pos(TARGET_RA_DEG, TARGET_DEC_DEG)
print("Value at TARGET_RA_DEG, TARGET_DEC_DEG (s):", val)


In [ ]:
ra = np.linspace(59.5, 60.5, 5)
dec = np.linspace(-37.5, -36.5, 5)
print("Along a Dec row, all RA:", hsp_exptime.get_values_pos(ra, dec[0]))


In [ ]:
# Far from survey data — expect sentinel / non-finite values
print("Off-footprint example:", hsp_exptime.get_values_pos(180.0, 0.0))


### Mini table: RA, Dec, pixel, exposure time

This pattern is handy when you are joining external catalogs to survey property maps.

In [ ]:
ra_tab = np.array([59.7, 60.0, 60.3])
dec_tab = np.array([-37.2, -37.0, -36.8])
pix_tab = ra_dec_to_nested_healpix(ra_tab, dec_tab, nside)
val_tab = hsp_exptime.get_values_pos(ra_tab, dec_tab)

hdr = f'{"RA (deg)":>10} {"Dec (deg)":>10} {"HEALPix":>12} {"t_exp (s)":>12}'
print(hdr)
print("-" * len(hdr))
for r, d, p, v in zip(ra_tab, dec_tab, pix_tab, val_tab):
    print(f"{r:10.4f} {d:10.4f} {int(p):12d} {float(v):12.3f}")


## 8.0 Inspect the map object

HealSparse stores a coarse coverage layer plus sparse high-resolution cells. Degrading collapses what needs to be deserialized from disk, which is why reads feel lighter.

In [ ]:
print(hsp_exptime)


## 9.0 Matplotlib: a small RA/Dec region

`pcolormesh` simply shows the values **sampled on the grid you choose**; it does not magically increase map resolution when you zoom. Keep grid sizes modest when maps are large.

In [ ]:
ra = np.linspace(330.0, 360.0, 120)
dec = np.linspace(-30.0, 0.0, 120)
x, y = np.meshgrid(ra, dec)
values = hsp_exptime.get_values_pos(x, y)

values[values <= 0] = np.nan

fig, ax = plt.subplots(figsize=(8, 5))
pcm = ax.pcolormesh(x, y, values, shading="auto")
ax.set_xlabel("Right Ascension (deg)")
ax.set_ylabel("Declination (deg)")
ax.set_title(f"i-band exposure time, NSIDE={hsp_exptime.nside_sparse}")
fig.colorbar(pcm, ax=ax, label="Exposure time (s)")
plt.show()


## 10.0 Projected maps with `skyproj`

`McBrydeSkyproj` with `lon_0` near the survey longitude is a convenient default for Rubin wide-area figures. The first panel uses automatic scaling. The second zooms and sets color limits from percentiles **inside the patch** so seconds of exposure time remain readable.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sp = skyproj.McBrydeSkyproj(ax=ax, lon_0=65.0)
sp.draw_hspmap(hsp_exptime)
sp.draw_colorbar(label=f"i-band exposure time (s), NSIDE={hsp_exptime.nside_sparse}")
plt.show()


If you switch to `%matplotlib widget`, use the toolbar zoom and watch the status line for longitude, latitude, and value. Call `sp.set_autorescale(False)` if you want to freeze colorbar limits while zooming.